# Large Langage Model Automation example

**Author** : Lucile Lapray

**Date** : May 2025

**Need to:**

- Verify your phone number in your profile
- Go to Settings > Turn on Internet
- Go to Settings > Accelerator > Set to GPU100
- Run all
- Then go to Run > Restart & Clear Cell Outputs
- Run all again

## 1. LLM package install and import

### 1.1. Package installation

In [ ]:
!pip install torch transformers accelerate bitsandbytes

In [ ]:
!pip install --upgrade transformers

### 1.2. Library import

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer
import torch
from tqdm import tqdm
import pandas as pd

## 2. Model loading

In [ ]:
model_name = "unsloth/phi-4-unsloth-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

## 3. Data import

In [ ]:
df = pd.read_csv('/kaggle/input/prostitution-movies-sample/prostitution_movies_sample.csv')
df.head()

## 4. Text generation

In [ ]:
# Wrap llm generation into a function
def generation(prompt) :
  model.generation_config.pad_token_id = tokenizer.pad_token_id

  messages = [
      {"role": "user", "content": prompt}
  ]
  input_tensor = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
  outputs = model.generate(input_tensor.to(model.device), max_new_tokens = 1000)

  result = tokenizer.decode(outputs[0][input_tensor.shape[1]:], skip_special_tokens=True)

  return result

### 4.1. Annotation of positivity

In [ ]:
# Annotate the synopsis for positivity
annotation_positivity = list()
for _, movie in tqdm(df.iterrows(), total = len(df), desc = "Annotating positivity of movies") :
  prompt = f"""Here\'s a movie synopsis from the movie '{movie['name']}' : '{movie['summary_text']}'.
  \\n\\On a scale form -1 to 1, how positive do you think this movie is?
  -1 means a very negative movie, and 1 means a very positive movie.
  Justify briefly, then finish your message in the following format: \'X\'"""
  annotation_positivity.append(generation(prompt))

In [ ]:
# Extract labels (scale from -1 to 1 of positivity)
positivity = []
for response in annotation_positivity:
    label = response.strip().split()[-1]
    positivity.append(label)
print(positivity)

### 4.2. Annotation of romanticization of prostitution

In [ ]:
# Annotate the synopsis for prostitution romanticization
annotation_romanticization = list()
for _, movie in tqdm(df.iterrows(), total = len(df), desc = "Annotating positivity of movies"):
  prompt = f"""Here\'s a movie synopsis from the movie '{movie['name']}' : '{movie['summary_text']}'.
  \\n\\On a scale form 0 to 10, how much does this movie romanticize prostitution?
  Justify briefly, then finish your message in the following format: \'Romanticization score : X\'."""
  annotation_romanticization.append(generation(prompt))

In [ ]:
# Extract labels (scale from -1 to 1 of positivity)
romanticization = []
for response in annotation_romanticization:
    label = response.strip().split()[-1].replace('.', '')
    romanticization.append(label)
print(romanticization)

## 5. Data saving

In [ ]:
df['annot_positivity'] = annotation_positivity
df['positivity'] = positivity
df['annot_romanticization'] = annotation_romanticization
df['romanticization'] = romanticization

In [ ]:
display(df)
df.to_csv('romanticization_movies_llm.csv', index=False)